# Lab 1: Creating Your First Agent with Tools

In this lab, you will create an AI agent using the Microsoft Agent Framework and give it a tool that looks up movie information. Agents combine an LLM with callable tools to take actions — but the agent must decide when a tool is relevant, which means it can miss opportunities to use its tools on broad or exploratory queries.

## What you will learn

- How to create an `OpenAIResponsesClient`
- How to define a tool as a plain Python function
- How to create an agent with `client.as_agent()`
- How to run the agent with streaming output
- The limitation of tools (reactive, not automatic)

## Setup

First, load the environment variables from your `.env` file.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Verify your OpenAI key is set
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

Import the required modules from the Microsoft Agent Framework and Pydantic.

In [ ]:
from typing import Annotated
from agent_framework.openai import OpenAIResponsesClient
from pydantic import Field

## Define Movie Data

Create some hardcoded movie data from the Neo4j recommendations dataset. This simulates what will later come from Neo4j automatically via context providers — the agent won't need hardcoded data once it can query the knowledge graph directly.

In [ ]:
MOVIES = {
    "INCEPTION": {
        "title": "Inception",
        "director": "Christopher Nolan",
        "year": "2010",
        "genres": ["Science Fiction", "Thriller"],
        "plot_summary": "A skilled thief who steals corporate secrets through dream-sharing technology is given a chance to erase his criminal record by planting an idea in a target's subconscious.",
    },
    "THE MATRIX": {
        "title": "The Matrix",
        "director": "Lana Wachowski, Lilly Wachowski",
        "year": "1999",
        "genres": ["Science Fiction", "Action"],
        "plot_summary": "A computer hacker learns about the true nature of his reality and his role in the war against its controllers.",
    },
    "PULP FICTION": {
        "title": "Pulp Fiction",
        "director": "Quentin Tarantino",
        "year": "1994",
        "genres": ["Crime", "Drama"],
        "plot_summary": "The lives of two mob hitmen, a boxer, a gangster and his wife intertwine in four tales of violence and redemption.",
    },
    "THE DARK KNIGHT": {
        "title": "The Dark Knight",
        "director": "Christopher Nolan",
        "year": "2008",
        "genres": ["Action", "Crime", "Drama"],
        "plot_summary": "When the menace known as the Joker wreaks havoc on Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.",
    },
}

## Define a Tool

In MAF, tools are plain Python functions with docstrings. The agent reads the function **name** and **docstring** to decide when to call the tool. No decorators or registration required.

Create a `get_movie_info` function that looks up a movie from the dictionary and returns its details.

In [ ]:
def get_movie_info(
    movie_title: Annotated[str, Field(description="The movie title to look up")]
) -> str:
    """Look up information about a movie including its director, year, genres, and plot summary."""
    key = movie_title.upper().strip()
    info = MOVIES.get(key)
    if not info:
        for k, v in MOVIES.items():
            if key in k or k in key:
                info = v
                break
    if info:
        return (
            f"Title: {info['title']}\n"
            f"Director: {info['director']}\n"
            f"Year: {info['year']}\n"
            f"Genres: {', '.join(info['genres'])}\n"
            f"Plot: {info['plot_summary']}"
        )
    available = ", ".join(m["title"] for m in MOVIES.values())
    return f"Movie '{movie_title}' not found. Available movies: {available}"

## Create and Run the Agent

Use `client.as_agent()` to create a stateless agent with a name, instructions, and tools. The agent has no memory of previous turns and no access to external data beyond its tools — it can only use information the LLM already knows or that a tool returns.

In [ ]:
client = OpenAIResponsesClient()

agent = client.as_agent(
    name="movie-info-agent",
    instructions=(
        "You are a helpful movie assistant. Use your tool to look up "
        "movie information when asked about a specific movie. If the "
        "user asks about a movie not in your database, let them know "
        "which movies are available."
    ),
    tools=[get_movie_info],
)

query = "Tell me about Inception"
print(f"User: {query}\n")
print("Answer: ", end="", flush=True)
async for update in agent.run(query, stream=True):
    if update.text:
        print(update.text, end="", flush=True)
print()

## The Limitation of Tools

Try asking the agent a question that does not name a specific movie. The agent may or may not call `get_movie_info` — it depends on whether the LLM decides the tool is relevant to a general question.

Tools are **reactive**: the agent must choose to call them. For broad or exploratory queries, it often does not.

In [ ]:
query = "What are some good sci-fi movies about time travel?"
print(f"User: {query}\n")
print("Answer: ", end="", flush=True)
async for update in agent.run(query, stream=True):
    if update.text:
        print(update.text, end="", flush=True)
print()

## Summary

You created an agent with a `get_movie_info` tool that looks up movie information. The agent uses the function name and docstring to decide when to call the tool.

**The limitation:** Tools are reactive. The agent must decide to call them, which means it might skip relevant context for general queries.

**What's next:** In the next lab, you will explore **context providers** — MAF's mechanism for automatically injecting relevant knowledge into every agent interaction, regardless of how the question is phrased.